# Open-XTTS-Trainer: End-to-End 3-Stage Training & Proof Verification Demo

This notebook runs an end-to-end training verification on **5 minutes of clean human speech** across all 3 stages of **XTTS v2**:
1. **Stage 2:** Text Tokenizer & Vocabulary Extension for New Scripts + Autoregressive GPT Backbone Training (Real DVAE Code Cross-Entropy)
2. **Stage 3:** Latent HiFi-GAN Vocoder Fine-Tuning (True Temporal Latents + Pretrained Weights)
3. **Stage 1:** Discrete VAE (DVAE) Audio Tokenizer Fine-Tuning at native 22,050 Hz
4. **Synthesis & Proof Export:** Audio generation from trained checkpoints + 1-click export of verified logs (`open_xtts_proof_logs.zip`).

## 1. Setup Environment & Install Dependencies

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Check GPU availability
!nvidia-smi

# Install dependencies and Idiap's Coqui TTS fork
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!pip install -q torchcodec
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .
!mkdir -p logs checkpoints xtts_base

## 2. Download Pretrained XTTS-v2 Base Checkpoints & 5-Min Speech Dataset

In [ ]:
# 1. Download official base checkpoints (vocab, config, dvae, model)
!wget -q -O xtts_base/vocab.json https://huggingface.co/coqui/XTTS-v2/raw/main/vocab.json
!wget -q -O xtts_base/config.json https://huggingface.co/coqui/XTTS-v2/raw/main/config.json
!wget -q -O xtts_base/dvae.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth
!wget -q -O xtts_base/model.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth

# 2. Choose speech data source:
# Default: Ultra-fast 10-Second Smoke Test (4 clips, entire notebook finishes in ~2 mins)
!python -m xtts_trainer.speech_data --num_samples 4 --duration 3.0 --output_dir sample_audio

# Option B: Use your own 10s-30s audio recording (upload my_voice.wav to Colab, then uncomment):
# !python -m xtts_trainer.speech_data --input_file my_voice.wav --output_dir sample_audio

# Option C: Full 5-minute training corpus (100 clips, for final full training):
# !python -m xtts_trainer.speech_data --minutes 5.0 --output_dir sample_audio

!ls -lh sample_audio


## 3. Stage 2: GPT Alignment & Vocabulary Extension

In [ ]:
# 1. Sample target text containing new language script
with open("sample_target.txt", "w", encoding="utf-8") as f:
    f.write("ދިވެހި ބަސް. สวัสดี. مرحبا. नमस्ते")

# 2. Extend tokenizer vocab (automatically resizes text_embedding & text_head)
!python xtts_trainer/stage2_gpt/vocab_extender.py --base_vocab xtts_base/vocab.json --target_text sample_target.txt --output_vocab xtts_base/vocab_extended.json

# 3. Fine-Tune Autoregressive GPT Backbone with true DVAE acoustic target cross-entropy
# Smoke test: 2 epochs (~15s on T4 GPU). Increase to 15-20 epochs for full production training.
!python xtts_trainer/stage2_gpt/train.py \
    --audio_dir sample_audio \
    --vocab_path xtts_base/vocab_extended.json \
    --checkpoint_dir xtts_base \
    --output_dir checkpoints/gpt \
    --epochs 2 \
    --batch_size 2 \
    --device cuda 2>&1 | tee logs/stage2_gpt.log


## 4. Stage 3: Latent HiFi-GAN Vocoder Fine-Tuning

In [ ]:
# 1. Extract true per-frame temporal GPT latents & speaker embeddings (~15s on T4 GPU)
!python xtts_trainer/stage3_hifigan/extract_latents.py \
    --audio_dir sample_audio \
    --output_dir extracted_latents \
    --checkpoint_dir xtts_base \
    --device cuda 2>&1 | tee logs/extract_latents.log

# 2. Fine-tune Latent HiFi-GAN Vocoder starting from pretrained base weights
# Smoke test: 2 epochs (~15s on T4 GPU). Increase to 20-25 epochs for full production training.
!python xtts_trainer/stage3_hifigan/train.py \
    --audio_dir sample_audio \
    --latents_dir extracted_latents \
    --checkpoint_dir xtts_base \
    --output_dir checkpoints/hifigan \
    --batch_size 2 \
    --epochs 2 \
    --device cuda 2>&1 | tee logs/stage3_hifigan.log


## 5. Stage 1: Discrete VAE (DVAE) Fine-Tuning

In [ ]:
# Run native DVAE fine-tuning at 22,050 Hz
# Smoke test: 2 epochs (~10s on T4 GPU). Increase to 20-25 epochs for full production training.
!python xtts_trainer/stage1_dvae/train.py \
    --audio_dir sample_audio \
    --output_dir checkpoints/dvae \
    --continue_path xtts_base/dvae.pth \
    --batch_size 2 \
    --epochs 2 \
    --device cuda 2>&1 | tee logs/stage1_dvae.log


## 6. Test Speech Synthesis with Custom Trained Weights

In [ ]:
import os
import glob
import soundfile as sf
import torch
import IPython.display as ipd
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from xtts_trainer.stage3_hifigan.inference import synthesize_audio

# Load model using fine-tuned Stage 2 GPT checkpoint
config = XttsConfig()
if os.path.exists("xtts_base/config.json"):
    config.load_json("xtts_base/config.json")

model = Xtts(config)
model.load_checkpoint(
    config,
    checkpoint_path="checkpoints/gpt/model.pth",
    vocab_path="checkpoints/gpt/vocab.json",
    eval=True
)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    model.cuda()

# Select reference audio for voice cloning (prefer non-silent active speech)
audio_files = sorted(glob.glob("sample_audio/*.wav"))
speaker_wav = None
for f in ["sample_audio/speech_003.wav", "sample_audio/speech_002.wav", "sample_audio/speech_001.wav"]:
    if os.path.exists(f):
        speaker_wav = f
        break
if not speaker_wav and audio_files:
    speaker_wav = audio_files[0]
if not speaker_wav:
    raise FileNotFoundError("No audio reference file found in sample_audio/ directory.")
print(f"Cloning voice using reference: {speaker_wav}")

test_prompt = "Artificial intelligence is rapidly transforming how we communicate and build open technology every day."
hifi_ckpt_path = "checkpoints/hifigan/hifigan.pth"

# -------------------------------------------------------------
# PROOF TEST 1: Direct Stage 3 Vocoder Reconstruction from Real Latents
# -------------------------------------------------------------
print("\n" + "="*60)
print("PROOF 1: Stage 3 Direct Decoding from Extracted Latent File")
print("="*60)
selected_latent = None
for l in ["latents/speech_003.pt", "latents/speech_002.pt", "latents/speech_001.pt"]:
    if os.path.exists(l):
        selected_latent = l
        break
if not selected_latent:
    pts = sorted(glob.glob("latents/*.pt"))
    if pts:
        selected_latent = pts[0]

if selected_latent and os.path.exists(hifi_ckpt_path):
    synthesize_audio(
        checkpoint_path=hifi_ckpt_path,
        latent_path=selected_latent,
        output_path="stage3_direct_recon.wav",
        device=device,
    )
    print(f"Playing Stage 3 direct reconstruction ({selected_latent}):")
    display(ipd.Audio("stage3_direct_recon.wav"))
else:
    print("Stage 3 checkpoint or latent not found for direct test.")

# -------------------------------------------------------------
# PROOF TEST 2: Synthesis with Base Pretrained Vocoder (Baseline)
# -------------------------------------------------------------
print("\n" + "="*60)
print("PROOF 2: Synthesis with Base Pretrained HiFi-GAN Vocoder")
print("="*60)
outputs_base = model.synthesize(
    test_prompt,
    speaker_wav=speaker_wav,
    language="en",
)
sf.write("cloned_speech_base.wav", outputs_base["wav"], 24000)
print("Playing baseline synthesis (base vocoder):")
display(ipd.Audio("cloned_speech_base.wav"))

# -------------------------------------------------------------
# PROOF TEST 3: Synthesis with Trained Stage 3 Vocoder Injected
# -------------------------------------------------------------
print("\n" + "="*60)
print("PROOF 3: Synthesis with Trained Stage 3 HiFi-GAN Vocoder")
print("="*60)
if os.path.exists(hifi_ckpt_path):
    hifi_ckpt = torch.load(hifi_ckpt_path, map_location=device)
    if "hifigan_decoder" in hifi_ckpt:
        model.hifigan_decoder.load_state_dict(hifi_ckpt["hifigan_decoder"], strict=False)
    elif "generator" in hifi_ckpt:
        model.hifigan_decoder.waveform_decoder.load_state_dict(hifi_ckpt["generator"], strict=False)
    print(f"Successfully swapped in trained Stage 3 weights from {hifi_ckpt_path}!")

outputs_s3 = model.synthesize(
    test_prompt,
    speaker_wav=speaker_wav,
    language="en",
)
sf.write("cloned_speech.wav", outputs_s3["wav"], 24000)
sf.write("cloned_speech_stage3.wav", outputs_s3["wav"], 24000)
print("Playing synthesis using trained Stage 3 HiFi-GAN vocoder:")
display(ipd.Audio("cloned_speech.wav"))


## 7. Generate Verification Proof Report & 1-Click Download

In [ ]:
# 1. Run automated proof compiler across all execution logs
!python scripts/generate_proof_report.py --logs_dir logs --checkpoints_dir checkpoints --audio_file cloned_speech.wav

# 2. Render Markdown proof table directly in Colab
from IPython.display import Markdown, display
with open("logs/training_proof_report.md", "r") as f:
    display(Markdown(f.read()))

# 3. Trigger 1-click browser download of the complete proof archive
try:
    from google.colab import files
    print("Downloading proof archive (open_xtts_proof_logs.zip)...")
    files.download('open_xtts_proof_logs.zip')
except Exception:
    print("Archive available at: open_xtts_proof_logs.zip")
